In [1]:
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

In [2]:
# === 샘플 문서 준비 ===
document = """Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다.
어댑터즈의 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성됩니다.
5단 분석법은 기술 용어를 일반 명사와 고유 명사로 나누어 설명하고,
사용 이유와 방법, 다른 기술과의 비교까지 체계적으로 정리하는 방식입니다.
어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다."""

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

# === 출력 스키마 정의 ===
# LLM이 반환할 응답의 구조를 Pydantic 모델로 정의합니다.
# 각 필드의 description은 LLM에게 "이 필드에 무엇을 넣어야 하는지" 알려줍니다.
class DocumentAnalysis(BaseModel):
    title: str = Field(description="문서의 제목")
    summary: str = Field(description="문서의 핵심 내용을 2~3문장으로 요약")
    keywords: list[str] = Field(description="문서의 핵심 키워드 3~5개")
    category: str = Field(description="문서의 분야 (예: 인공지능, 웹 개발, 인프라 등)")

class SentimentAnalysis(BaseModel):
    sentiment: Literal["positive", "negative", "neutral"] = Field(description="문장의 감정")
    confidence: str = Field(description="판단 근거를 한 줄로 설명")

In [4]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# === Chat Model 생성 ===
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=GOOGLE_API_KEY
)

# === with_structured_output으로 스키마 바인딩 ===
# 모델에 스키마를 바인딩하면, 응답이 Pydantic 객체로 반환됩니다.
structured_model = model.with_structured_output(DocumentAnalysis)

# === 프롬프트 구성 ===
prompt = ChatPromptTemplate.from_messages([
    ("system", "다음 문서를 분석하세요."),
    ("human", "{document}")
])

# === LCEL 체인 구성 ===
# StrOutputParser 없이, prompt → structured_model로 바로 연결합니다.
chain = prompt | structured_model

In [5]:
# === 체인 실행 ===
result = chain.invoke({"document": document})

# === 결과 확인 ===
# result는 DocumentAnalysis 객체입니다.
print(f"타입: {type(result).__name__}")
print(f"제목: {result.title}")
print(f"요약: {result.summary}")
print(f"키워드: {result.keywords}")
print(f"분야: {result.category}")

타입: DocumentAnalysis
제목: Adapterz(어댑터즈) 소개
요약: Adapterz(어댑터즈)는 스타트업코드에서 운영하는 개발 교재 서빙 서비스입니다. 모든 교재는 5단 분석법이라는 독자적인 교수법으로 작성되며, 이 방법은 기술 용어를 체계적으로 분석하고 설명합니다. 어댑터즈는 인공지능, 데이터 분석, 웹 개발, 인프라 분야의 교재를 제공합니다.
키워드: ['Adapterz', '5단 분석법', '개발 교재', '스타트업코드', '기술 교육']
분야: IT 교육


In [6]:
# === 같은 스키마, 다른 문서 ===
another_doc = """LangChain은 LLM 애플리케이션 개발을 위한 오픈소스 프레임워크입니다.
프롬프트 템플릿, Chat Model, 출력 파서, 검색기 같은 컴포넌트를
LCEL 파이프 연산자로 연결하여 체인을 구성합니다."""

result2 = chain.invoke({"document": another_doc})
print(f"제목: {result2.title}")
print(f"키워드: {result2.keywords}")

제목: LangChain 개요
키워드: ['LangChain', 'LLM', '프레임워크', 'LCEL', '체인']
